# 2.4 Machine Learning

Las fases previas de limpieza, análisis exploratorio y feature engineering prepararon los datos para entrenar modelos de aprendizaje supervisado. En este notebook dividiremos los datos, entrenaremos modelos y evaluaremos su capacidad para generalizar a pasajeros no vistos.

## Importar paquetes

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score

# Para guardar el modelo
import pickle

## Carga de datos

In [2]:
df = pd.read_csv('./data/titanic_procesado.csv')
print(f'Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas')
df.head()

Dimensiones: 891 filas y 8 columnas


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,1,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,0,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


## División de datos

`X` contiene las variables predictoras y `y` contiene la variable objetivo `Survived`. Después reservamos el 20 % de los registros para evaluar el modelo con datos no utilizados durante el entrenamiento.

In [3]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [4]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convertir los DataFrames y Series en arreglos de NumPy
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [6]:
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape} | y_test:  {y_test.shape}')
print(f'Supervivencia total: {y.mean():.2%}')
print(f'Supervivencia en entrenamiento: {y_train.mean():.2%}')
print(f'Supervivencia en prueba: {y_test.mean():.2%}')

X_train: (712, 7) | y_train: (712,)
X_test:  (179, 7) | y_test:  (179,)
Supervivencia total: 38.38%
Supervivencia en entrenamiento: 38.34%
Supervivencia en prueba: 38.55%


> **Nota:** `stratify=y` mantiene aproximadamente la misma proporción de pasajeros sobrevivientes y fallecidos en los conjuntos de entrenamiento y prueba. `random_state=42` permite reproducir exactamente la misma división.

## Entrenamiento

Definimos once algoritmos y sus espacios de hiperparámetros. `GridSearchCV` probará cada combinación mediante validación cruzada y seleccionará el mejor estimador de cada familia.

In [7]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(random_state=42),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(random_state=42),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(random_state=42),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(random_state=42),
        'parametros': {'n_estimators': [10, 100]}
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {'n_neighbors': [3, 5, 7]}
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(random_state=42),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(random_state=42),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3],
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {'alpha': [0.1, 1.0, 10.0]}
    }
}

In [8]:
# Inicializar las variables para almacenar resultados
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })
    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados en un DataFrame
metricas = pd.DataFrame(puntajes_modelos).sort_values(
    'Precisión', ascending=False
)

print('Rendimiento de los modelos de clasificación')
print(metricas.round(2).to_string(index=False))
print('---------------------------------------------------')
print('MEJOR MODELO DE CLASIFICACIÓN')
print(f'Modelo: {mejor_modelo}')
print(f'Precisión: {mejor_precision:.2f}')
print(f'Mejores hiperparámetros: {mejor_estimador.get_params()}')

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

...

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

Rendimiento de los modelos de clasificación
                             Modelo  Precisión
                Regresión Logística       0.81
  Clasificador de Gradient Boosting       0.80
               Clasificador XGBoost       0.79
Clasificador de Vectores de Soporte       0.79
   Clasificador K-Nearest Neighbors       0.79
  Clasificador de Árbol de Decisión       0.78
 Clasificador de Bosques Aleatorios       0.78
              Clasificador AdaBoost       0.78
           Clasificador Naive Bayes       0.78
                  Clasificador LGBM       0.77
                         GaussianNB       0.74
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Regresión Logística
Precisión: 0.81
Mejores hiperparámetros: {'C': 10, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 100, 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'saga', 'tol': 0.0001, 'verbose': 0, 'warm_start

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

## Entrenamiento sin GridSearch

Entrenamos una regresión logística con sus hiperparámetros predeterminados para observar el flujo básico de ajuste, predicción y evaluación.

In [9]:
# Estas importaciones ya se realizaron al inicio; se repiten como referencia
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Creamos el modelo de regresión logística
model = LogisticRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Realizamos predicciones con el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluamos el modelo usando precisión
accuracy = accuracy_score(y_test, y_pred)

print(f'Precisión del modelo: {accuracy:.2f}')

Precisión del modelo: 0.79


### Regresión logística con hiperparámetros manuales

In [10]:
model = LogisticRegression(
    C=0.5,                   # Valor inverso de la fuerza de regularización
    penalty='l2',           # Regularización Ridge
    solver='lbfgs',         # Algoritmo de optimización
    max_iter=200,           # Número máximo de iteraciones
    class_weight='balanced' # Ajustar pesos de las clases
)

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f'Precisión del modelo: {accuracy:.2f}')

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Precisión del modelo: 0.78


## Inferencia

Utilizamos el mejor estimador para predecir la supervivencia de un registro con las mismas siete características y en el mismo orden utilizado durante el entrenamiento.

In [11]:
print('Primer registro de X_train:')
print(X_train[0])
print(f'Valor real en y_train: {y_train[0]}')

Primer registro de X_train:
[1.         1.         0.50725294 0.         0.         0.600375
 1.        ]
Valor real en y_train: 1


In [12]:
# reshape(1, -1) representa un pasajero con siete características
nuevos_datos = np.array(X_train[0]).reshape(1, -1)
prediccion = mejor_estimador.predict(nuevos_datos)

print(f'Predicción: {prediccion}')
print('Resultado: sobrevivió' if prediccion[0] == 1 else 'Resultado: no sobrevivió')

Predicción: [0]
Resultado: no sobrevivió


## Guardar el modelo

In [13]:
import pickle

with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print('Modelo guardado en modelo.pkl')

Modelo guardado en modelo.pkl


In [14]:
# Verificar que el modelo guardado puede cargarse y predecir
with open('modelo.pkl', 'rb') as archivo_estimador:
    modelo_cargado = pickle.load(archivo_estimador)

prediccion_cargada = modelo_cargado.predict(nuevos_datos)
print(f'Predicción del modelo cargado: {prediccion_cargada}')

Predicción del modelo cargado: [0]


> **Importante:** `modelo.pkl` contiene solamente el estimador entrenado. Los datos nuevos deben recibir exactamente la misma limpieza, codificación, transformación y escalado aplicados en los notebooks anteriores. Para producción conviene integrar esas transformaciones y el modelo en un único `Pipeline` y guardar el pipeline completo. Además, nunca cargues archivos Pickle de fuentes no confiables.